In [10]:
# USE THIS WHEN RUNNING ON COLAB
"""import numpy as np
import pandas as pd
import kagglehub

# Download latest version
path = kagglehub.dataset_download("s1nghayush/data-dump")

print("Path to dataset files:", path)

df = pd.read_csv(path + '/ganjoo_ipl_mbm_bowl_final.csv')"""

'import numpy as np\nimport pandas as pd\nimport kagglehub\n\n# Download latest version\npath = kagglehub.dataset_download("s1nghayush/data-dump")\n\nprint("Path to dataset files:", path)\n\ndf = pd.read_csv(path + \'/ganjoo_ipl_mbm_bowl_final.csv\')'

In [6]:
# USE THIS WHEN RUNNING LOCALLY
import pandas as pd
import numpy as np

df = pd.read_csv(r"D:\ED\FIFS_IPLungit\data\mbm\all_league_bowl_final.csv")

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 161041 entries, 0 to 161040
Data columns (total 18 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Unnamed: 0     161041 non-null  int64  
 1   match_id       161041 non-null  int64  
 2   date           161041 non-null  object 
 3   inns           161041 non-null  int64  
 4   bowler         161041 non-null  object 
 5   bowler_id      161041 non-null  int64  
 6   bowl_team      161041 non-null  object 
 7   bat_team       161041 non-null  object 
 8   overs          161041 non-null  object 
 9   runs           161041 non-null  float64
 10  wickets        161041 non-null  int64  
 11  maidens        161041 non-null  int64  
 12  eco            161041 non-null  float64
 13  balls          161041 non-null  int64  
 14  dots           161041 non-null  int64  
 15  lbw_or_bowled  161041 non-null  int64  
 16  venue          161041 non-null  object 
 17  league         161041 non-nul

In [8]:
# Replace 'DNB' with NaN and convert 'overs' to float
df['overs'] = df['overs'].replace('DNB', np.nan).astype(float)

# Fill NaN values in 'overs' with 0 (if needed)
df['overs'] = df['overs'].fillna(0)

In [9]:
# Replace 'DNB' with NaN and convert 'overs' to float
df['overs'] = df['overs'].replace('DNB', np.nan).astype(float)

# Fill NaN values in 'overs' with 0
df['overs'] = df['overs'].fillna(0)

# Calculate dot ball points (+1 per 3 dot balls)
df['dot_ball_points'] = df['dots'] * 1

# Calculate wicket points (+25 per wicket)
df['wicket_points'] = df['wickets'] * 25

# Calculate bonus for LBW/Bowled (+8 per dismissal)
df['bonus_lbw_bowled'] = df['lbw_or_bowled'] * 8

# Wicket milestone bonuses
df['3_wicket_bonus'] = ((df['wickets'] >= 3) & (df['wickets'] < 4)).astype(int) * 4
df['4_wicket_bonus'] = ((df['wickets'] >= 4) & (df['wickets'] < 5)).astype(int) * 8
df['5_wicket_bonus'] = (df['wickets'] >= 5).astype(int) * 12

# Maiden over bonus (+4 per maiden over)
df['maiden_over_bonus'] = df['maidens'] * 12

# Economy rate points calculation function
def calculate_economy_points(row):
    if row['overs'] < 2:  # Minimum 5 overs condition
        return 0
    eco = row['eco']
    if eco < 5:
        return 6
    elif 5 <= eco < 5.99:
        return 4
    elif 6 <= eco < 7:
        return 2
    elif 10 <= eco < 11:
        return -2
    elif 11.01 <= eco < 12:
        return -4
    elif eco >= 12:
        return -6
    return 0  # Default case

# Apply economy points calculation
df['economy_rate_points'] = df.apply(calculate_economy_points, axis=1)

# Total fantasy points calculation
df['bowling_fp'] = (
    df['dot_ball_points'] +
    df['wicket_points'] +
    df['bonus_lbw_bowled'] +
    df['4_wicket_bonus'] +
    df['5_wicket_bonus'] +
    df['3_wicket_bonus'] +
    df['maiden_over_bonus'] +
    df['economy_rate_points']
)

# Drop unnecessary columns
df.drop(columns=[
    'dot_ball_points', 'wicket_points', 'bonus_lbw_bowled',
    '4_wicket_bonus', '5_wicket_bonus', '3_wicket_bonus', 'maiden_over_bonus', 'economy_rate_points'
], inplace=True)

In [10]:
df = df[['bowler_id', 'match_id', 'bowling_fp', 'date']]

In [11]:
min(df.date), max(df.date)

('2005-02-17', '2025-04-07')

In [12]:
df

,bowler_id,match_id,bowling_fp,date
0,6278,211028,4,2005-06-13
1,6565,211028,89,2005-06-13
2,6126,211028,27,2005-06-13
3,5392,211028,32,2005-06-13
4,4578,211028,29,2005-06-13
...,...,...,...,...
161036,820685,1476798,26,2025-03-23
161037,1229251,1476798,0,2025-03-23
161038,1206188,1476798,0,2025-03-23
161039,817069,1476798,0,2025-03-23


In [13]:
df.to_csv(r"D:\ED\FIFS_IPLungit\data\processed_data\all_league_bowl_fp.csv", index=False)